# Using PICASO for Roman Coronagraph Exoplanet Atmospheric Modeling

In this hands-on tutorial you will learn the basic PICASO procedures that can launch you into assessing a target’s observability and science prospects given Roman-CGI's specifications. 

We'll cover:
* Using PICASO to calculate optical spectra of substellar companions with both reflected light and thermal emission components
* Familiarity with the physical information encoded in these observations and in which regions of parameter space reflected light vs thermal emission dominates
* Familiarity with some relevant self-consistent 1d-RCE model grids for Roman-CGI targets
* Procedure for simulating noisy Roman-CGI observations and looping through a grid to find minimum $\chi^2$ model

It is expected that you attended the accompanying conceptual lecture given by Zarah and are comfortable working in python. Notebooks A-C are designed to take ~3 hours for a brand new PICASO user to work through. If you finish early, you are invited to apply your new skills towards a self-directed exploration of Roman-CGI science potential, or take a look at quickstart_climate.ipynb.

jump down to [colab set up](#colab) and [local installation](#local)

------------------------
## Roman-CGI Overview 

Roman-CGI is a NASA technology demonstrator flying with the flagship Roman Space Telescope. For the first time in space, it will use deformable mirrors and wave-front sensing and control to serve as a key stepping stone towards Habitable Worlds Observatory. Roman will observe at shorter wavelengths than we've been able to directly image any exoplanets before this.

**Roman-CGI Observing Modes:**

| Band |Mode |  Center | Width | Resolution | Data points |
|------|------|--------|--------|------------|-------------|
|  Band 1 |Photometric  | 573.8 nm | 9.8% | -- | 1 |
|  Band 3 |Spectroscopic |729.3 nm | 16.8% | R = 50 | ~9 |
|  Band 4 |Photometric  | 825.5 nm | 11.7%  |  -- | 1|

As part of its technology demonstration, Technology Threshold Requirement 5 (**TTR5**) states that Roman-CGI will make a measurement with SNR > 5 at a separation of 6-9 $\lambda$/D, with a contrast of 10^-7 in a 10% bandwidth at a central wavelength < 600 nm. The contrast and working angle relative to other facilities and possible targets is shown in panel (A) below. 


![alt text](Images/roman_contrast_curves_targets.png)

Roman-CGI will likely demonstrate TTR5 on a **>2400 K brown dwarf** companion to a main-sequence host star, HIP 71618 B. At this temperature, the light collected by Roman-CGI will be dominated by **thermal emission**. Roman-CGI will likely also carry out photometric and spectral observations of some additional young, hot, and widely orbiting substellar companions with a broad range of temperatures ~700-2400 K.

For the first time ever, Roman-CGI will aim to image **1-3 old+cold exoplanets** that reside near their host stars' snow-lines, true solar system gas giant analogues! These have been detected via radial velocity but never imaged. The light collected from them will be totally dominated by **reflected light** from their host star in Roman-CGI's wavelength range. 

The objects currently planned to make up part of Roman-CGI's observing schedule are marked with red outlines and placed in the context of other exoplanets in the NASA exoplanet archive in panel (B above). Some of their properties relevant for atmosphere modeling are summarized in the Table below.

### Table of Approximate Properties for Top Roman-CGI Targets

---

**RV planets (reflected-light targets)**

| Name | T_eq (K) | Mass (M_Jup) | log g (cgs)| Radius (R_Jup) | a (AU) | Star T_eff (K) | Star log g | Star Radius (R_sun) |
|------|------------|----------------|----------|-----------------|--------|------------|---------|-----------------|
| 14 Her b | ~142 | 8.5 ± 1.0 | — | — | 2.77 | 5270 | 4.5 * | 0.87 |
| ups And d | ~238 | 10.25 | — | — | 2.51 | 6160 | 4.1 * | 1.62 |
| eps Eri b | ~112 | 0.66 | — | — | 3.53 | 5084 | 4.6 * | 0.735 |
| 47 Uma c | ~165 | ≥0.54 † | — | — | 3.60 | 5872 | 4.30 | 1.21 |

---

We don't have radii and surface gravities for the RV planets yet. Something like https://github.com/chenjj2/forecaster could be a good way to pick reasonable radii based on their masses. Or you could adopt a radius based on evolutionary models.

**Directly imaged self-luminous companions**

| Name | T_eff (K) | Mass (M_Jup) | log g (cgs) | Radius (R_Jup) | a (AU) | Star T_eff (K) | Star log g | Star Radius (R_sun) |
|------|------------|----------------|----------|-----------------|--------|------------|---------|-----------------|
| HIP 71618 B | 2700 ± 100 | 60 (+27/−21) | 4.5 | 1.70 | 11.1 (+1.3/−1.0) | 9300 (+400/−500) | 4.3 * | ~2.0 * |
| HIP 54515 b | ~2000 | 17.7 (+7.6/−4.9) | — | — | ~25 | ~8200 * | ~4.2 * | ~1.8 * |
| beta Pic b | 1607 (+5/−6) | 9–13 | 4.46 (+0.02/−0.04) | 1.46 | ~9.2 | 8054 | 4.18 | 1.53 |
| kappa And b | 1791 ± 68 | 17.3 ± 1.8 | 4.35 ± 0.07 | 1.42 ± 0.06 | 55 | 10900 | 3.78 | — |
| HR 8799 c | ~1150 | ~9–10 | ~3.5–4.0 | ~1.2 | 38 | 7400 | 4.0 | ~1.44 * |
| HR 8799 d | ~1100 | ~9–10 | ~3.5–4.0 | ~1.2 | 24 | ″ | ″ | ″ |
| HR 8799 e | ~1200 | ~10 | ~3.5–4.0 | ~1.2 | 16.4 | ″ | ″ | ″ |


**CAUTION:** These quantities in both RL and SL tables include many rough estimates for your pedagogical purposes, not publication ready! 

------------------------
## PICASO Overview


![alt text](Images/phase_angle_disco.png)

- started as the first open-source reflected light post-processing tool
- grew out of legacy fortran codes, upgraded to object-oriented and modular pythonic design
- extensive additional tutorials and documentation available at: https://natashabatalha.github.io/picaso/tutorials.html#
- now includes most capabilities needed for modeling exoplanets or brown dwarfs:
  - reflected light spectra
  - thermal emission spectra
  - transit spectra
  - phase curves with 3d variation around planet
  - 1D RCE climate modeling, including irradiation, clouds, disequilibrium chemistry, photochemistry
  - companion virga package handles cloud calculations
  - built-in tools for grid-fitting
  - retrieval framework (coming soon)
  - GPU version of the code (coming soon)
- now the main code base for producing Sonora family of grid models, paired with evolutionary code as boundary conditions to get cooling curves
- latest PICASO version release was very recent! version 4: https://arxiv.org/abs/2602.22468
- ethos of open source reproducible science


--------------------
## <a id="local"></a> Option 1: Local "developer" installation of PICASO and virga for more extensive use

If you don't already have the **anaconda** platform installed for package and environment management, you can download it here: https://www.anaconda.com/download

#### Obtaining Tutorial Materials

Pick whatever working directory you want this tutorial to be in and either download the zip file of this github respository or git clone it. https://github.com/blacy/picaso_for_roman_coronagraph_school

#### Make a new virtual environment and activate it

`$ conda create -n picaso_v4 python=3.12`

`$ conda activate picaso_v4`

#### (optional) I like to also install my preferred code development environment

`$ conda install -c conda-forge jupyterlab`

#### Install the virga python package which handles clouds for PICASO

Notes:
- The `pip install -e .` command keeps PICASO and virga packages there in your current working directory, so you can easily take a look at the source code rather than squirreling them away in a hidden directory. A "developer" installation.
- If you don't want them hidden away with other python packages, you can instead follow installation instructions on the PICASO website: https://natashabatalha.github.io/picaso/installation.html for obtaining both PICASO and Virga. When it comes time to download **reference data**, jump back to the instructions provided below rather than for the generic installation.

`$ git clone https://github.com/natashabatalha/virga.git` 

`$ cd virga`

`$ pip install -e .`

`$ cd ../`

#### Depending on which order you install virga and PICASO, you can end up with some version incompatibilities of pandas and numpy, so let's make sure they are both set to versions that work with PICASO

`$ conda run -n picaso_v4 pip install "pandas==2.3.0"`

`$ conda run -n picaso_v4 pip install "numpy==1.26.4"`

#### Install the PICASO python package, now version 4

`$ git clone https://github.com/natashabatalha/picaso.git`

`$ cd picaso`

`$ pip install -e .`

#### Download the reference data  (skip to here if you didn't want to do a developer installation)

We've prepared a reference folder specifically for this workshop. The main difference is lower resolution opacities than you would need for
modeling anything above R~100 accurately. Download this zenodo repository: https://zenodo.org/records/18892896, unzip it and place it as a subdirectory inside the `picaso` directory, keeping the name `picaso-reference-romanschool`. Note you do NOT want to put it in the subdirectory `picaso/picaso/picaso-reference-romanschool`, just `picaso/picaso-reference-romanschool`.

#### Finally you can set your paths to reference data and test!

In [ ]:
# every time you use picaso,
# you need to set environment variables called 
# picaso_refdata and PYSYN_CDBS that point towards
# this data we just downloaded

import os
# ── SET YOUR PATHS HERE ──────────────────────────────────────────
PICASO_REFDATA = '/Users/briannalacy/Research/Roman_CPP/Picaso_Tutorial/picaso/picaso-reference-romanschool'
PYSYN_CDBS = PICASO_REFDATA+'/stellar_grids'
# ─────────────────────────────────────────────────────────────────
os.environ['picaso_refdata'] = PICASO_REFDATA
os.environ['PYSYN_CDBS'] = PYSYN_CDBS

Let's run a quick check of whether the download and the path-setting worked correctly...

In [ ]:
# finally check whether everything worked!
import picaso.data as data
data.check_environ()

from picaso import justdoit as jdi
from picaso import justplotit as jpi
from virga import justdoit as vj

We don't need `refrence/opacities/resortrebin`, `reference/opacities/resampled`, or the Vega spectrum for now, so if those are your only warnings, should be good to go.

### Success! 

------------------------
## <a id="colab"></a>  Option 2: Google Co Lab installation

Co Lab does not play nicely with the full set of opacities needed for this tutorial, so this is not the preferred method of working through it. But in the event PICASO is not happy on your personal machine you can still work through much of the material. The key difference is that we will only use a subset of the relevant wavelength range and molecular species. Methods/procedures are all the same, but any conclusions about spectral features and shape are suspect.

(instructions heavily inspired by the nice work done by https://nexsci.caltech.edu/workshop/2023/handson.shtml) 

#### Create a clean google account with 15 GB storage
To work throught the tutorials in google co-lab you will need:
- A new free Google account to maximize the available 15G storage:
https://www.google.com/account/about/
- To recover diskspace after running each setup notebook, be sure to empty the trash in your
google drive to get rid of the temporary download files. You may need to wait a bit or you can
re-login again to see the recovered space.

1) Log into your Google account
2) Navigate to Google Drive - either follow this link: https://drive.google.com/drive/my-drive OR
from your account click on the dot navigation and then click on the Drive icon.
3) Once in Drive, on the left side menu, there is a section called
“Storage” which will show how much space you have available. If you
have insufficient storage in your Google account, we suggest creating
a new account rather than purchasing storage. Note that if you hit the
limit of storage, it will affect your email usage, so it is better to create
a new account.


#### Add tutorial materials to your google drive

Upload the contents of this GitHub repository, except the README.md notebooks into your google drive. You may like to put them in folder in there named RomanSchoolColab or something of that sort. 

Steps to do this are: 

1. Download a zipped version of this repository from GitHub and unzip locally
2. Be logged in to your google account in a chrome window
3. navigate to https://colab.research.google.com/
4. click on the "upload" option on the right and pick one of the tutorial notebooks to upload
5. repeat steps 3+4 with other notebooks

They are all now stored in a folder named "Colab Notebooks" within your google drive.

#### Add our roman_tutorial_utils.py module to google drive

1. Make a new folder in your google drive named "RomanSchoolPICASO"
2. Add the file 'roman_tutorial_utils.py' from the downloaded GitHub repository to it by navigating to the folder, clicking  the "+ New" button on the upper right and selecting 'File upload'

#### Set up PICASO and virga in co lab environment

1. Open this notebook (A_introduction_and_setup.ipynb) in google Co Lab either finding it within https://colab.research.google.com/ or within https://drive.google.com/drive/my-drive
2. Uncomment and run the cells below within co-lab, you just cick the little right arrows by the upper left of each cell in order.

In [ ]:
reference_prefix_dir = 'RomanSchoolPICASO'

# Colab
import os

# Mount Google Drive
from google.colab import drive
drive.mount("/content/drive")

# Google top level drive dir
drive_dir = "/content/drive/MyDrive/"

# prefix dir directory path
reference_prefix_dir = os.path.join(drive_dir, reference_prefix_dir)

# Change working directory to the prefix_path
os.chdir(reference_prefix_dir)

The following cell took me ~10 minutes on the Villa Clythia wifi, alternatively obtain this from Arthur's hard drive and upload to google drive.

In [ ]:
# Download Picaso data for Roman School
!wget -O reference.tar.gz https://zenodo.org/records/18892896/files/picaso-lite-roman-google-colab.tar.gz
!tar xf reference.tar.gz
!rm reference.tar.gz

In [ ]:
# install Picaso with pip
!pip install picaso

In [ ]:
# set up the path to that reference data we just downloaded
PICASO_REFDATA = reference_prefix_dir+'/picaso-reference-romanschool'
PYSYN_CDBS = PICASO_REFDATA+'/stellar_grids'

os.environ['picaso_refdata'] = PICASO_REFDATA
os.environ['PYSYN_CDBS'] = PYSYN_CDBS

import picaso.data as data
data.check_environ()

In [ ]:
from picaso import justdoit as jdi
from picaso import justplotit as jpi
from virga import justdoit as vj

If you made it here without errors (warnings are ok), you are ready to move on to notebooks B and C. Again, bear in mind that all spectra are being computed with some missing opacities and for procedural PICASO pedagogy only, not science!

### Woohoo!

----------------------

Proceed to B_picaso_for_optical_spectra.ipynb

Remember that you'll need the following few lines at the start of a script or notebook every time you want to use PICASO, edited to the appropriate file paths for your own computer.

In [3]:
import os
# ── SET YOUR PATHS HERE ──────────────────────────────────────────
PICASO_REFDATA = '/Users/briannalacy/Research/Roman_CPP/Picaso_Tutorial/picaso/picaso-reference-romanschool'
PYSYN_CDBS = PICASO_REFDATA+'/stellar_grids'
# ─────────────────────────────────────────────────────────────────
os.environ['picaso_refdata'] = PICASO_REFDATA
os.environ['PYSYN_CDBS'] = PYSYN_CDBS

from picaso import justdoit as jdi
from picaso import justplotit as jpi
from virga import justdoit as vj

If working in Google Co Lab you also need to mount your drive and install picaso, so your first cell should look like: 

In [ ]:
# THIS CELL FOR GOOGLE CO LAB ONLY

from google.colab import drive
drive.mount("/content/drive")
drive_dir = "/content/drive/MyDrive/"

import os
# if you stored things in drive with names from tutorial A, there 
# should be no need to edit this
reference_prefix_dir = 'RomanSchoolPICASO'
reference_prefix_dir = os.path.join(drive_dir, reference_prefix_dir)
os.chdir(reference_prefix_dir)

PICASO_REFDATA = reference_prefix_dir+'/picaso-reference-romanschool'
PYSYN_CDBS = PICASO_REFDATA+'/stellar_grids'
os.environ['picaso_refdata'] = PICASO_REFDATA
os.environ['PYSYN_CDBS'] = PYSYN_CDBS

!pip install picaso

from picaso import justdoit as jdi
from picaso import justplotit as jpi
from virga import justdoit as vj